# Cross-y analysis

In previous experiments, the fitted u only correspond to a single observation y. This notebook now tries to analyse how a certain feature could affect several observations simultaneously.

In [25]:
import sys
import os
import pickle

sys.path.insert(0, os.path.abspath(".."))

import numpy as np

from sklearn.preprocessing import StandardScaler

from src.function_library import build_function_library, select_top_power_features, power_features, build_power_library
from src.sparse_interp import sparse_ee_interpretation, save_sparse_result, load_sparse_result, sparse_predict, refine_sparse_result

from npeet import entropy_estimators as ee
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [27]:
def ee_objective(coeffs, X, y):
    coeffs = np.asarray(coeffs, dtype=float)
    coeffs /= np.linalg.norm(coeffs)
    u = X @ coeffs
    return ee.mi(y, u)

def scipy_objective(coeffs, X, y):
    return -ee_objective(coeffs, X, y)

In [2]:
# loading data
data = np.load(r"D:\Law\25-26\research intern\final_data.npz")
print(data.files)

X_final = data["param"]

param_names = [
    "C_p",
    "Za_p",
    "R_p",
    "Emax_rv",
    "Emin_rv",
    "C_s",
    "Za_s",
    "R_s",
    "Emax_lv",
    "Emin_lv"
]

v_lv_final = data["v_lv"]
v_rv_final = data["v_rv"]

y_v_lv_max_final = np.max(v_lv_final, axis=1)
y_v_lv_min_final = np.min(v_lv_final, axis=1)

y_v_rv_max_final = np.max(v_rv_final, axis=1)
y_v_rv_min_final = np.min(v_rv_final, axis=1)

y_v_lv_mean_final = np.mean(v_lv_final, axis=1)
y_v_rv_mean_final = np.mean(v_rv_final, axis=1)

y_v_combined = np.column_stack([
    y_v_lv_max_final,
    y_v_lv_min_final,
    y_v_lv_mean_final,
    y_v_rv_max_final,
    y_v_rv_min_final,
    y_v_rv_mean_final,
])

y_v_lv_combined = np.column_stack([
    y_v_lv_max_final,
    y_v_lv_min_final,
    y_v_lv_mean_final,
])
y_v_rv_combined = np.column_stack([
    y_v_rv_max_final,
    y_v_rv_min_final,
    y_v_rv_mean_final,
])

y_v_max_combined = np.column_stack([
    y_v_lv_max_final,
    y_v_rv_max_final,
])
y_v_min_combined = np.column_stack([
    y_v_lv_min_final,
    y_v_rv_min_final,
])
y_v_mean_combined = np.column_stack([
    y_v_lv_mean_final,
    y_v_rv_mean_final,
])


['p_lv', 'p_rv', 'p_pa', 'v_lv', 'v_rv', 'q_av', 'q_mv', 'q_pv', 'param']


In [31]:
# load old sampling
with open("../notebook/final_data_split.pkl", "rb") as f:
    final_data_split = pickle.load(f)

final_train_idx = final_data_split["train_idx"]
final_test_idx = final_data_split["test_idx"]

X_final_train = X_final[final_train_idx]
X_final_test = X_final[final_test_idx]

y_v_lv_max_final_train = y_v_lv_max_final[final_train_idx]
y_v_lv_max_final_test = y_v_lv_max_final[final_test_idx]

y_v_lv_min_final_train = y_v_lv_min_final[final_train_idx]
y_v_lv_min_final_test = y_v_lv_min_final[final_test_idx]

y_v_rv_max_final_train = y_v_rv_max_final[final_train_idx]
y_v_rv_max_final_test = y_v_rv_max_final[final_test_idx]

y_v_rv_min_final_train = y_v_rv_min_final[final_train_idx]
y_v_rv_min_final_test = y_v_rv_min_final[final_test_idx]

y_v_lv_mean_final_train = y_v_lv_mean_final[final_train_idx]
y_v_rv_mean_final_train = y_v_rv_mean_final[final_train_idx]

y_v_lv_mean_final_test = y_v_lv_mean_final[final_test_idx]
y_v_rv_mean_final_test = y_v_rv_mean_final[final_test_idx]

# LV combined
y_v_lv_combined_train = y_v_lv_combined[final_train_idx]
y_v_lv_combined_test = y_v_lv_combined[final_test_idx]

# RV combined
y_v_rv_combined_train = y_v_rv_combined[final_train_idx]
y_v_rv_combined_test = y_v_rv_combined[final_test_idx]

# Max combined
y_v_max_combined_train = y_v_max_combined[final_train_idx]
y_v_max_combined_test = y_v_max_combined[final_test_idx]

# Min combined
y_v_min_combined_train = y_v_min_combined[final_train_idx]
y_v_min_combined_test = y_v_min_combined[final_test_idx]

# Mean combined
y_v_mean_combined_train = y_v_mean_combined[final_train_idx]
y_v_mean_combined_test = y_v_mean_combined[final_test_idx]

# All combined
y_v_combined_train = y_v_combined[final_train_idx]
y_v_combined_test = y_v_combined[final_test_idx]

In [43]:
Theta_v, feature_names_v = build_function_library(
    X_final_train,
    param_names
)

In [10]:
def analyze_joint_mi(
    Theta,
    feature_names,
    Y,
):
    Theta = np.asarray(Theta)
    Y = np.asarray(Y)

    results = []

    for i, name in enumerate(feature_names):

        x = Theta[:, i].reshape(-1, 1)

        mi = ee.mi(
            x,
            Y
        )

        results.append({
            "feature": name,
            "mi": mi
        })

    results.sort(
        key=lambda x: x["mi"],
        reverse=True
    )

    return results

def print_joint_mi_results(results_dict):
    features = []

    for results in results_dict.values():
        for result in results:
            if result["feature"] not in features:
                features.append(result["feature"])

    # Store MI values
    mi_table = {}

    for feature in features:
        mi_table[feature] = {}

    for target_name, results in results_dict.items():

        for result in results:

            feature = result["feature"]
            mi = result["mi"]

            mi_table[feature][target_name] = mi

    # Print header
    target_names = list(results_dict.keys())

    header = f"{'Feature':35s}"

    for target_name in target_names:
        header += f"{target_name:>12s}"

    print(header)
    print("-" * len(header))

    # Sort by first target
    first_target = target_names[0]

    features.sort(
        key=lambda feature:
            mi_table[feature].get(first_target, float("-inf")),
        reverse=True
    )

    # Print rows
    for feature in features:

        row = f"{feature:35s}"

        for target_name in target_names:

            mi = mi_table[feature].get(
                target_name,
                float("nan")
            )

            row += f"{mi:12.6f}"

        print(row)

    return mi_table

In [44]:
results_v_lv = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_lv_combined_train
)
results_v_rv = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_rv_combined_train
)
results_v_max = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_max_combined_train
)
results_v_min = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_min_combined_train
)
results_v_mean = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_mean_combined_train
)
results_v = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_combined_train
)

In [45]:
results_dict = {
    "All": results_v,
    "LV": results_v_lv,
    "RV": results_v_rv,
    "Max": results_v_max,
    "Min": results_v_min,
    "Mean": results_v_mean,
}

mi_table = print_joint_mi_results(
    results_dict
)

Feature                                     All          LV          RV         Max         Min        Mean
-----------------------------------------------------------------------------------------------------------
Emax_lv^2                              1.120369    1.481069    0.446843    1.255675    1.263873    1.416129
R_s                                    1.008995    0.748420    0.846021    0.846349    0.626001    0.659056
R_p                                    0.982722    0.966240    0.983696    0.763639    0.571966    0.608844
Emax_rv*R_s                            0.966601    0.567405    0.992880    0.880436    0.680517    0.657374
R_p*Emax_rv                            0.949883    0.571848    0.913582    0.905768    0.734057    0.827470
R_p*Emax_lv                            0.929283    1.035529    0.419093    0.925288    0.555193    0.754386
Emin_rv*R_s                            0.836123    0.661925    0.811607    0.608962    0.561837    0.574101
R_s*Emin_lv                 